# Vanilla RAG Lab: Mars Landing Report Q&A

Build a simple RAG system that answers questions about a fictional report of humanity's first Mars landing.

**What you'll learn:**
- Chunk documents with multiple strategies (fixed, recursive, semantic)
- Embed and index chunks into ChromaDB
- Build a RAG agent with LangChain + LangGraph
- Evaluate RAG quality with retrieval and answer metrics
- Improve retrieval with hybrid search (BM25 + dense) and cross-encoder reranking
- Compare vanilla vs hybrid RAG with precision@k, recall@k, nDCG, and LLM-as-judge

In [29]:
import os
import logging
from dotenv import load_dotenv

load_dotenv("../.env")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
)
logger = logging.getLogger("rag_lab")

logger.info("OpenAI API key configured: %s", "Yes" if os.getenv("OPENAI_API_KEY") else "No")
logger.info("OpenAI API key configured: %s", "Yes" if os.getenv("LANGCHAIN_API_KEY") else "No")

2026-04-08 21:29:36,836 [INFO] rag_lab - OpenAI API key configured: Yes
2026-04-08 21:29:36,837 [INFO] rag_lab - OpenAI API key configured: Yes


## Step 1: Load the Document

Read the Mars landing report that serves as our knowledge base.

In [19]:
with open("first-time-on-mars.txt", "r") as f:
    document_text = f.read()

logger.info("Loaded document: %d characters", len(document_text))
print(document_text[:500] + "\n...")

2026-04-08 21:10:17,809 [INFO] rag_lab - Loaded document: 8526 characters


**Dateline: July 18, 2050 — Jezero Crater, Mars**

For the first time in human history, the red dust of Mars bears the unmistakable imprint of a human footprint.

At 09:42 Coordinated Mars Time, Commander Elena Navarro descended the final rung of the lander *Astraeus* and stepped onto the Martian surface, pausing only briefly before uttering words that will likely be remembered for generations:

> “We are no longer visitors of the sky—we are a species that has arrived.”

Behind her, the horizon 
...


## Step 2: Chunk the Document

Split the document into smaller chunks for embedding. Three strategies are available — change the `CHUNKING_STRATEGY` flag to compare them:

| Strategy | Description |
|---|---|
| `fixed` | Fixed-size character splits with overlap |
| `recursive` | Recursive splitting that respects natural text boundaries (paragraphs, sections) |
| `semantic` | Groups text by embedding similarity — chunks are semantically coherent |

In [20]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

# =============================================================
#  CHANGE THIS FLAG TO SWITCH CHUNKING STRATEGY
CHUNKING_STRATEGY = "fixed"  # "fixed" | "recursive" | "semantic"
# =============================================================

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100 #TODO: Change chunk size and overlap to see the effect on the results


def chunk_document(text: str, strategy: str) -> list[str]:
    """Split text into chunks using the selected strategy."""
    logger.info("Chunking strategy=%s  size=%d  overlap=%d", strategy, CHUNK_SIZE, CHUNK_OVERLAP)

    if strategy == "fixed":
        splitter = CharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separator="\n",
        )
    elif strategy == "recursive":
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n---\n", "\n### ", "\n\n", "\n", " "],
        )
    elif strategy == "semantic":
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        splitter = SemanticChunker(embeddings, breakpoint_threshold_type="percentile")
    else:
        raise ValueError(f"Unknown chunking strategy: {strategy}")

    return splitter.split_text(text)


chunks = chunk_document(document_text, CHUNKING_STRATEGY)
logger.info("Produced %d chunks", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n{'=' * 60}")
    print(f"Chunk {i}  ({len(chunk)} chars)")
    print(f"{'=' * 60}")
    print(chunk[:300] + ("..." if len(chunk) > 300 else ""))

2026-04-08 21:12:31,433 [INFO] rag_lab - Chunking strategy=fixed  size=500  overlap=100
2026-04-08 21:12:31,440 [INFO] rag_lab - Produced 23 chunks



Chunk 0  (471 chars)
**Dateline: July 18, 2050 — Jezero Crater, Mars**
For the first time in human history, the red dust of Mars bears the unmistakable imprint of a human footprint.
At 09:42 Coordinated Mars Time, Commander Elena Navarro descended the final rung of the lander *Astraeus* and stepped onto the Martian surf...

Chunk 1  (498 chars)
Behind her, the horizon stretched in muted tones of rust and amber, broken only by distant ridges and the faint silhouette of ancient riverbeds—silent witnesses to a world that once held water, and perhaps, life. Above, the sky glowed a pale, dusty orange, thinner and more fragile than anything seen...

Chunk 2  (370 chars)
---
### The Mission: A Journey Decades in the Making
The mission, officially named *Aurora-1*, is the culmination of over thirty years of international collaboration, technological innovation, and relentless ambition. Unlike previous robotic missions, which paved the way with data and imagery, *Auro...

Chunk 3  (313 chars)


## Step 3: Embed and Index into ChromaDB

Convert chunks into vector embeddings using OpenAI's `text-embedding-3-small` model and store them in a local ChromaDB collection with metadata.

In [21]:
import chromadb
from langchain_chroma import Chroma
from langchain_core.documents import Document

PERSIST_DIR = "./chroma_db"
COLLECTION_NAME = "mars_landing_report"

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Clear previous collection so re-runs with different strategies start fresh
chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
existing = [c.name for c in chroma_client.list_collections()]
if COLLECTION_NAME in existing:
    chroma_client.delete_collection(COLLECTION_NAME)
    logger.info("Deleted existing collection '%s'", COLLECTION_NAME)

documents = [
    Document(
        page_content=chunk,
        metadata={
            "source": "first-time-on-mark.txt",
            "chunk_index": i,
            "chunking_strategy": CHUNKING_STRATEGY,
        },
    )
    for i, chunk in enumerate(chunks)
]

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIR,
)

logger.info("Indexed %d chunks into ChromaDB (collection='%s')", len(documents), COLLECTION_NAME)

# Quick sanity check
results = vectorstore.similarity_search("Who landed on Mars?", k=3)
print("Sanity check — top 3 results for 'Who landed on Mars?':\n")
for i, doc in enumerate(results):
    print(f"  [{i + 1}] chunk_index={doc.metadata['chunk_index']}")
    print(f"      {doc.page_content}\n")

2026-04-08 21:13:45,006 [INFO] rag_lab - Deleted existing collection 'mars_landing_report'
2026-04-08 21:13:46,156 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:13:46,284 [INFO] rag_lab - Indexed 23 chunks into ChromaDB (collection='mars_landing_report')
2026-04-08 21:13:46,670 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Sanity check — top 3 results for 'Who landed on Mars?':

  [1] chunk_index=4
      The landing itself was a feat of precision engineering. Mars’ thin atmosphere has long posed a challenge: too thick to ignore, too thin to rely on for safe descent. The *Astraeus* lander employed a hybrid system of heat shielding, supersonic parachutes, and retro-thrusters, guided by AI-assisted navigation that adjusted in real time to atmospheric conditions.

  [2] chunk_index=0
      **Dateline: July 18, 2050 — Jezero Crater, Mars**
For the first time in human history, the red dust of Mars bears the unmistakable imprint of a human footprint.
At 09:42 Coordinated Mars Time, Commander Elena Navarro descended the final rung of the lander *Astraeus* and stepped onto the Martian surface, pausing only briefly before uttering words that will likely be remembered for generations:
> “We are no longer visitors of the sky—we are a species that has arrived.”

  [3] chunk_index=11
      Walking on Mars is an experi

## Step 4: Build the RAG Agent

Create a LangGraph agent equipped with a retriever tool that searches the ChromaDB vector store to answer questions.

In [22]:
from langchain_openai import ChatOpenAI
from langchain_core.tools.retriever import create_retriever_tool
from langchain.agents import create_agent

#TODO: always search the vector store for the most relevant chunks instead of using tool

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

retriever_tool = create_retriever_tool(
    retriever,
    name="mars_report_search",
    description=(
        "Search the Mars landing report for information about the Aurora-1 mission, "
        "its crew, scientific findings, spacecraft, and life on Mars. "
        "Use this tool to find facts before answering any question."
    ),
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

SYSTEM_PROMPT = (
    "You are a knowledgeable assistant that answers questions about humanity's "
    "first Mars landing. Always use the mars_report_search tool to retrieve "
    "relevant information before answering. Base your answers strictly on the "
    "retrieved context. If the answer is not in the retrieved documents, say so."
)

agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=SYSTEM_PROMPT,
)

logger.info("RAG agent created with retriever tool")

2026-04-08 21:16:09,260 [INFO] rag_lab - RAG agent created with retriever tool


In [23]:
test_questions = [
    "Who was the first person to step on Mars?",
    "What was the name of the mission and when did it launch?",
    "How many crew members were there and what were their nationalities?",
    "What scientific discoveries were made on Mars?",
    "What challenges did the crew face living on Mars?",
]

for question in test_questions:
    print(f"\n{'=' * 70}")
    print(f"Q: {question}")
    print(f"{'=' * 70}")
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content
    print(f"A: {answer}")


Q: Who was the first person to step on Mars?


2026-04-08 21:16:26,152 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:16:26,791 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:16:29,468 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The first person to step on Mars was Commander Elena Navarro. She descended from the lander *Astraeus* and made her historic footprint on the Martian surface at 09:42 Coordinated Mars Time on July 18, 2050.

Q: What was the name of the mission and when did it launch?


2026-04-08 21:16:31,532 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:16:31,822 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:16:33,797 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The mission was officially named *Aurora-1*, and it launched in late 2049.

Q: How many crew members were there and what were their nationalities?


...

A: The *Aurora-1* mission crew consists of six members, each from different nationalities:

1. **Commander Elena Navarro** - Spain
2. **Captain Lucas Andrade** - Brazil
3. **Dr. Hannah Weiss** - Germany

Unfortunately, the report does not provide the names or nationalities of the remaining three crew members.

Q: What scientific discoveries were made on Mars?


2026-04-08 21:17:01,317 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:17:01,524 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:17:07,274 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The scientific discoveries made on Mars during the Aurora-1 mission include:

1. **Geological Insights**: The terrain of Mars is described as a mosaic of fine dust and jagged rocks, shaped by billions of years of geological processes. The crew has begun collecting samples for analysis, which may provide further insights into the planet's history.

2. **Potential Signs of Life**: Dr. Okoye noted that early observations suggest Mars was once more dynamic and potentially more hospitable than previously believed. However, it is still too early to draw definitive conclusions about the presence of life.

3. **Health and Environmental Studies**: The mission is also focused on understanding the effects of Mars' environment on human health. This includes monitoring radiation exposure, which is a significant concern due to Mars' lack of a global magnetic field. The crew's health is being assessed daily to study the impacts of reduced gravity and increased radiation.

These findings indicate t

2026-04-08 21:17:08,904 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:17:09,374 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:17:14,667 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The crew faced several significant challenges while living on Mars:

1. **Extreme Temperatures**: Mars experiences dramatic temperature fluctuations, with relatively mild conditions during the day and extreme cold at night.

2. **Radiation Exposure**: The lack of a global magnetic field on Mars leaves its surface vulnerable to cosmic rays, posing a risk to the crew's health. Although the habitat is equipped with shielding, long-term exposure to radiation remains a concern.

3. **Reduced Gravity**: Mars has about 38% of Earth's gravity, which affects how the crew moves and interacts with their environment. They had to adapt to a different way of walking, which was described as having a peculiar, dreamlike gait.

4. **Bulky Space Suits**: The crew had to wear bulky suits designed to protect them from extreme temperatures, radiation, and the near-vacuum atmosphere, which constrained their movements.

5. **Isolation and Silence**: The Martian environment is characterized by overwhelming

## Step 5: Evaluate RAG Quality

Evaluate both **retrieval** and **generation** quality using a test dataset with ground-truth answers.

**Retrieval metrics:**
- **Hit Rate** — fraction of questions where the correct context appears in top-k results
- **MRR (Mean Reciprocal Rank)** — average of `1 / rank` for the first relevant chunk

**Answer metrics (LLM-as-judge):**
- **Faithfulness** — does the answer stick to the retrieved context (no hallucination)?
- **Correctness** — does the answer match the ground truth?

In [24]:
eval_dataset = [
    {
        "question": "Who was the commander of the Aurora-1 mission?",
        "ground_truth": "Commander Elena Navarro from Spain.",
        "keywords": ["Elena Navarro", "Spain"],
    },
    {
        "question": "When did humans first land on Mars?",
        "ground_truth": "Humans first landed on Mars on July 18, 2050.",
        "keywords": ["July 18, 2050"],
    },
    {
        "question": "What was the name of the lander?",
        "ground_truth": "The lander was named Astraeus.",
        "keywords": ["Astraeus"],
    },
    {
        "question": "What is the gravity on Mars compared to Earth?",
        "ground_truth": "Mars gravity is about 38 percent of Earth's gravity.",
        "keywords": ["38%", "38 percent"],
    },
    {
        "question": "What was Dr. Malik Okoye's role on the mission?",
        "ground_truth": "Dr. Malik Okoye was the astrobiologist and mission science lead from Nigeria.",
        "keywords": ["astrobiologist", "science lead"],
    },
    {
        "question": "What were the early scientific findings on Mars?",
        "ground_truth": "Preliminary scans identified complex carbon compounds in certain sediment layers.",
        "keywords": ["carbon compounds"],
    },
    {
        "question": "Where did the Aurora-1 mission land on Mars?",
        "ground_truth": "The mission landed at Jezero Crater, believed to be an ancient lakebed.",
        "keywords": ["Jezero Crater"],
    },
    {
        "question": "What was the name of the interplanetary vessel?",
        "ground_truth": "The interplanetary vessel was named Odyssey.",
        "keywords": ["Odyssey"],
    },
    {
        "question": "How long was the journey from Earth to Mars?",
        "ground_truth": "The journey lasted just under seven months.",
        "keywords": ["seven months"],
    },
    {
        "question": "What is the planned total duration of the Aurora-1 mission?",
        "ground_truth": "The mission is scheduled to last approximately 18 months including the return journey.",
        "keywords": ["18 months"],
    },
]

logger.info("Evaluation dataset: %d questions", len(eval_dataset))

2026-04-08 21:25:11,813 [INFO] rag_lab - Evaluation dataset: 10 questions


In [25]:
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

JUDGE_TEMPLATE = """Evaluate the following RAG-generated answer on two criteria.
Score each from 1 (worst) to 5 (best).

Question: {question}
Ground Truth: {ground_truth}
Retrieved Context:
{context}
Generated Answer: {answer}

Criteria:
- Faithfulness: Does the answer ONLY use information present in the Retrieved Context (no hallucination)?
- Correctness: Does the answer correctly address the question when compared to the Ground Truth?

Respond in EXACTLY this format (just the two lines, numbers only):
Faithfulness: <score>
Correctness: <score>"""

eval_results = []

for idx, item in enumerate(eval_dataset):
    question = item["question"]
    ground_truth = item["ground_truth"]
    keywords = item["keywords"]

    # --- Retrieval ---
    retrieved_docs = retriever.invoke(question)
    retrieved_texts = [doc.page_content for doc in retrieved_docs]
    combined_context = "\n---\n".join(retrieved_texts)

    hit = any(
        any(kw.lower() in txt.lower() for txt in retrieved_texts)
        for kw in keywords
    )

    reciprocal_rank = 0.0
    for rank, txt in enumerate(retrieved_texts, start=1):
        if any(kw.lower() in txt.lower() for kw in keywords):
            reciprocal_rank = 1.0 / rank
            break

    # --- Generation ---
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    # --- LLM-as-judge ---
    judge_prompt = JUDGE_TEMPLATE.format(
        question=question,
        ground_truth=ground_truth,
        context=combined_context[:3000],
        answer=answer,
    )
    judge_response = judge_llm.invoke(judge_prompt)
    judge_text = judge_response.content

    faithfulness_score = 0
    correctness_score = 0
    for line in judge_text.strip().split("\n"):
        if line.lower().startswith("faithfulness"):
            try:
                faithfulness_score = int(line.split(":")[1].strip())
            except (ValueError, IndexError):
                pass
        elif line.lower().startswith("correctness"):
            try:
                correctness_score = int(line.split(":")[1].strip())
            except (ValueError, IndexError):
                pass

    eval_results.append({
        "question": question,
        "ground_truth": ground_truth,
        "answer": answer,
        "hit": hit,
        "reciprocal_rank": reciprocal_rank,
        "faithfulness": faithfulness_score,
        "correctness": correctness_score,
    })

    logger.info(
        "[%d/%d] %s  hit=%s  rr=%.2f  faith=%d  correct=%d",
        idx + 1, len(eval_dataset), question[:40],
        hit, reciprocal_rank, faithfulness_score, correctness_score,
    )

logger.info("Evaluation complete")

2026-04-08 21:26:09,206 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:26:11,160 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:26:11,481 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:26:13,359 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:26:14,716 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:26:14,717 [INFO] rag_lab - [1/10] Who was the commander of the Aurora-1 mi  hit=True  rr=1.00  faith=5  correct=5
2026-04-08 21:26:14,909 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:26:15,880 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:26:16,189 [INFO] httpx - HTTP Request: POST

In [26]:
print(f"\n{'=' * 90}")
print(f"{'EVALUATION RESULTS':^90}")
print(f"{'=' * 90}")
print(f"\n{'Question':<50} {'Hit':>4} {'RR':>5} {'Faith':>6} {'Corr':>6}")
print(f"{'-' * 50} {'-' * 4} {'-' * 5} {'-' * 6} {'-' * 6}")

for r in eval_results:
    q_display = (r['question'][:47] + '...') if len(r['question']) > 50 else r['question']
    hit_symbol = 'Y' if r['hit'] else 'N'
    print(
        f"{q_display:<50} {hit_symbol:>4} {r['reciprocal_rank']:>5.2f} "
        f"{r['faithfulness']:>4}/5  {r['correctness']:>4}/5"
    )

hit_rate = sum(r["hit"] for r in eval_results) / len(eval_results)
mrr = sum(r["reciprocal_rank"] for r in eval_results) / len(eval_results)
avg_faith = sum(r["faithfulness"] for r in eval_results) / len(eval_results)
avg_correct = sum(r["correctness"] for r in eval_results) / len(eval_results)

print(f"\n{'=' * 90}")
print(f"  Chunking Strategy  : {CHUNKING_STRATEGY}")
print(f"  Hit Rate           : {hit_rate:.0%}")
print(f"  MRR                : {mrr:.3f}")
print(f"  Avg Faithfulness   : {avg_faith:.2f} / 5")
print(f"  Avg Correctness    : {avg_correct:.2f} / 5")
print(f"{'=' * 90}")
print(f"\nTip: Re-run the notebook with a different CHUNKING_STRATEGY to compare results.")


                                    EVALUATION RESULTS                                    

Question                                            Hit    RR  Faith   Corr
-------------------------------------------------- ---- ----- ------ ------
Who was the commander of the Aurora-1 mission?        Y  1.00    5/5     5/5
When did humans first land on Mars?                   Y  1.00    5/5     5/5
What was the name of the lander?                      Y  1.00    5/5     5/5
What is the gravity on Mars compared to Earth?        Y  1.00    5/5     5/5
What was Dr. Malik Okoye's role on the mission?       Y  1.00    5/5     5/5
What were the early scientific findings on Mars?      Y  0.20    5/5     5/5
Where did the Aurora-1 mission land on Mars?          Y  0.20    5/5     5/5
What was the name of the interplanetary vessel?       Y  1.00    5/5     5/5
How long was the journey from Earth to Mars?          N  0.00    5/5     2/5
What is the planned total duration of the Auror...    Y  1.00 

In [27]:
#TODO: add this evaluation score to langfuse/langsmith

## Step 6: Metadata-Enriched RAG with Citations

So far, our chunks carry minimal metadata (`source`, `chunk_index`, `chunking_strategy`). In production RAG systems, **rich metadata** unlocks two powerful capabilities:

| Capability | Description |
|---|---|
| **Filtered retrieval** | Narrow search to specific sections, topics, or speakers — instead of searching the entire corpus |
| **Source citation** | Trace every fact in the answer back to a specific section and chunk, so users can verify claims |

In this step we'll:
1. Parse the document into named sections and enrich each chunk with metadata (section title, people mentioned, whether it contains a direct quote)
2. Show how metadata filters improve retrieval precision
3. Build a citation-aware RAG agent that attributes every fact to its source

In [30]:
import re

METADATA_COLLECTION = "mars_landing_metadata"

CREW_MEMBERS = [
    "Elena Navarro", "Malik Okoye", "Aiko Tanaka",
    "Lucas Andrade", "Hannah Weiss", "Ravi Patel",
]

raw_sections = re.split(r'\n---\n', document_text)

metadata_chunks = []
for section_text in raw_sections:
    section_text = section_text.strip()
    if not section_text:
        continue

    header_match = re.match(r'###\s+(.+)', section_text)
    section_title = header_match.group(1).strip() if header_match else "Introduction"

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", " "],
    )
    section_chunks = splitter.split_text(section_text)

    for chunk_text in section_chunks:
        people_in_chunk = [
            name for name in CREW_MEMBERS if name.split()[-1] in chunk_text
        ]
        has_quote = bool(re.search(r'[\u201c\u201d]', chunk_text))

        metadata_chunks.append({
            "content": chunk_text,
            "metadata": {
                "source": "first-time-on-mark.txt",
                "section": section_title,
                "people_mentioned": ", ".join(people_in_chunk) if people_in_chunk else "none",
                "has_quote": has_quote,
                "chunk_index": len(metadata_chunks),
            },
        })

logger.info(
    "Parsed %d chunks with rich metadata across %d sections",
    len(metadata_chunks),
    len(set(m["metadata"]["section"] for m in metadata_chunks)),
)

# Index into a dedicated collection
existing = [c.name for c in chroma_client.list_collections()]
if METADATA_COLLECTION in existing:
    chroma_client.delete_collection(METADATA_COLLECTION)
    logger.info("Deleted existing collection '%s'", METADATA_COLLECTION)

metadata_documents = [
    Document(page_content=item["content"], metadata=item["metadata"])
    for item in metadata_chunks
]

metadata_vectorstore = Chroma.from_documents(
    documents=metadata_documents,
    embedding=embedding_model,
    collection_name=METADATA_COLLECTION,
    persist_directory=PERSIST_DIR,
)
logger.info("Indexed %d chunks into '%s'", len(metadata_documents), METADATA_COLLECTION)

print(f"\n{'Chunk':>5}  {'Section':<48} {'People':<30} {'Quote'}")
print(f"{'-'*5}  {'-'*48} {'-'*30} {'-'*5}")
for item in metadata_chunks:
    m = item["metadata"]
    print(
        f"{m['chunk_index']:>5}  {m['section']:<48} "
        f"{m['people_mentioned']:<30} {m['has_quote']}"
    )

2026-04-08 21:33:17,247 [INFO] rag_lab - Parsed 25 chunks with rich metadata across 9 sections
2026-04-08 21:33:17,259 [INFO] rag_lab - Deleted existing collection 'mars_landing_metadata'
2026-04-08 21:33:18,990 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:33:19,086 [INFO] rag_lab - Indexed 25 chunks into 'mars_landing_metadata'



Chunk  Section                                          People                         Quote
-----  ------------------------------------------------ ------------------------------ -----
    0  Introduction                                     Elena Navarro                  True
    1  Introduction                                     none                           False
    2  The Mission: A Journey Decades in the Making     none                           False
    3  The Mission: A Journey Decades in the Making     none                           False
    4  The Mission: A Journey Decades in the Making     none                           False
    5  The Mission: A Journey Decades in the Making     none                           False
    6  The Crew: Six Pioneers of a New Frontier         Elena Navarro                  False
    7  The Crew: Six Pioneers of a New Frontier         Malik Okoye, Aiko Tanaka       False
    8  The Crew: Six Pioneers of a New Frontier         Lucas Andrade,

In [31]:
query = "Who is the science lead?"

print(f"{'=' * 80}")
print(f"UNFILTERED: '{query}'")
print(f"{'=' * 80}")
for doc in metadata_vectorstore.similarity_search(query, k=3):
    print(f"  Section: {doc.metadata['section']}")
    print(f"  People:  {doc.metadata['people_mentioned']}")
    print(f"  Text:    {doc.page_content[:150]}...\n")

print(f"{'=' * 80}")
print(f"FILTERED by section = 'The Crew: Six Pioneers of a New Frontier'")
print(f"{'=' * 80}")
for doc in metadata_vectorstore.similarity_search(
    query, k=3,
    filter={"section": "The Crew: Six Pioneers of a New Frontier"},
):
    print(f"  Section: {doc.metadata['section']}")
    print(f"  People:  {doc.metadata['people_mentioned']}")
    print(f"  Text:    {doc.page_content[:150]}...\n")

quote_query = "What did the crew say about Mars?"
print(f"{'=' * 80}")
print(f"FILTERED by has_quote = True: '{quote_query}'")
print(f"{'=' * 80}")
for doc in metadata_vectorstore.similarity_search(
    quote_query, k=3,
    filter={"has_quote": True},
):
    print(f"  Section: {doc.metadata['section']}")
    print(f"  Quote:   {doc.metadata['has_quote']}")
    print(f"  Text:    {doc.page_content[:150]}...\n")

UNFILTERED: 'Who is the science lead?'


2026-04-08 21:34:05,040 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


  Section: Science in Action: Searching for Signs of Life
  People:  Malik Okoye, Aiko Tanaka
  Text:    ### Science in Action: Searching for Signs of Life

Central to the mission is the search for biosignatures—chemical or structural indicators of past l...

  Section: The Crew: Six Pioneers of a New Frontier
  People:  Malik Okoye, Aiko Tanaka
  Text:    * **Dr. Malik Okoye (Nigeria)** — Astrobiologist and mission science lead. His research focuses on extremophiles—organisms that survive in harsh envir...

  Section: The Crew: Six Pioneers of a New Frontier
  People:  Lucas Andrade, Hannah Weiss
  Text:    * **Captain Lucas Andrade (Brazil)** — Pilot and systems specialist. Responsible for navigation, landing operations, and maintaining the integrity of ...

FILTERED by section = 'The Crew: Six Pioneers of a New Frontier'


2026-04-08 21:34:05,374 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


  Section: The Crew: Six Pioneers of a New Frontier
  People:  Malik Okoye, Aiko Tanaka
  Text:    * **Dr. Malik Okoye (Nigeria)** — Astrobiologist and mission science lead. His research focuses on extremophiles—organisms that survive in harsh envir...

  Section: The Crew: Six Pioneers of a New Frontier
  People:  Lucas Andrade, Hannah Weiss
  Text:    * **Captain Lucas Andrade (Brazil)** — Pilot and systems specialist. Responsible for navigation, landing operations, and maintaining the integrity of ...

  Section: The Crew: Six Pioneers of a New Frontier
  People:  Elena Navarro
  Text:    ### The Crew: Six Pioneers of a New Frontier

The *Aurora-1* crew consists of six individuals, each representing a different discipline and, in many w...

FILTERED by has_quote = True: 'What did the crew say about Mars?'


2026-04-08 21:34:05,718 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


  Section: Science in Action: Searching for Signs of Life
  Quote:   True
  Text:    “It’s too early to draw conclusions,” Dr. Okoye emphasized during a press briefing. “But what we’re seeing suggests that Mars was once far more dynami...

  Section: First Impressions: A World Both Familiar and Alien
  Quote:   True
  Text:    Walking on Mars is an experience unlike any other. The planet’s gravity—about 38% of Earth’s—allows for longer strides and a peculiar, almost dreamlik...

  Section: Introduction
  Quote:   True
  Text:    **Dateline: July 18, 2050 — Jezero Crater, Mars**

For the first time in human history, the red dust of Mars bears the unmistakable imprint of a human...



In [32]:
from langchain_core.tools import tool


@tool
def search_mars_report_with_citations(query: str) -> str:
    """Search the Mars landing report. Returns passages with source metadata for citation."""
    docs = metadata_vectorstore.similarity_search(query, k=5)
    results = []
    for doc in docs:
        meta = doc.metadata
        header = f"[Section: {meta['section']}, Chunk: {meta['chunk_index']}]"
        if meta["people_mentioned"] != "none":
            header += f" (mentions: {meta['people_mentioned']})"
        results.append(f"{header}\n{doc.page_content}")
    return "\n\n---\n\n".join(results)


CITATION_SYSTEM_PROMPT = (
    "You are a knowledgeable assistant that answers questions about humanity's "
    "first Mars landing. Always use the search_mars_report_with_citations tool "
    "to retrieve relevant information before answering.\n\n"
    "IMPORTANT CITATION RULES:\n"
    "- Each retrieved passage starts with [Section: <name>, Chunk: <id>].\n"
    "- You MUST cite sources after each fact using the format "
    "[Section: <name>].\n"
    "- Example: 'Commander Navarro stepped onto Mars at 09:42 CMT "
    "[Section: Introduction].'\n"
    "- If the answer is not in the retrieved documents, say so."
)

citation_agent = create_agent(
    model=llm,
    tools=[search_mars_report_with_citations],
    system_prompt=CITATION_SYSTEM_PROMPT,
)
logger.info("Citation-aware RAG agent created")

citation_questions = [
    "Who was the first person to step on Mars and what did they say?",
    "What are the names and roles of all crew members?",
    "What scientific equipment is being used on Mars?",
    "What challenges does the crew face with radiation?",
]

for question in citation_questions:
    print(f"\n{'=' * 80}")
    print(f"Q: {question}")
    print(f"{'=' * 80}")
    response = citation_agent.invoke(
        {"messages": [{"role": "user", "content": question}]}
    )
    print(f"A: {response['messages'][-1].content}")

2026-04-08 21:35:06,406 [INFO] rag_lab - Citation-aware RAG agent created



Q: Who was the first person to step on Mars and what did they say?


2026-04-08 21:35:08,448 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:35:09,125 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:35:11,194 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The first person to step on Mars was Commander Elena Navarro, who made her historic descent onto the Martian surface at 09:42 Coordinated Mars Time on July 18, 2050. Upon stepping onto the red planet, she paused and declared:

> “We are no longer visitors of the sky—we are a species that has arrived.” [Section: Introduction].

Q: What are the names and roles of all crew members?


2026-04-08 21:35:12,270 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:35:12,459 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:35:19,441 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The crew of the *Aurora-1* mission to Mars consists of six members, each with distinct roles:

1. **Commander Elena Navarro (Spain)** — A veteran astronaut and aerospace engineer, she leads the mission and is known for her calm precision [Section: The Crew: Six Pioneers of a New Frontier].

2. **Captain Lucas Andrade (Brazil)** — The pilot and systems specialist responsible for navigation, landing operations, and maintaining the mission's critical systems [Section: The Crew: Six Pioneers of a New Frontier].

3. **Dr. Hannah Weiss (Germany)** — The medical officer and bioengineer who oversees crew health and leads experiments on human adaptation to Martian conditions [Section: The Crew: Six Pioneers of a New Frontier].

4. **Engineer Ravi Patel (India)** — A robotics and infrastructure expert managing the deployment of autonomous rovers and the construction of the mission's temporary habitat [Section: The Crew: Six Pioneers of a New Frontier].

5. **Dr. Malik Okoye (Nigeria)** — The 

2026-04-08 21:35:20,517 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:35:20,890 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:35:21,946 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:35:22,201 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:35:24,583 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The report does not provide specific details about the scientific equipment or instruments being used on Mars. It mentions that the crew has begun collecting samples for analysis, but does not specify the types of scientific instruments involved in this process [Section: First Impressions: A World Both Familiar and Alien]. 

If you have any other questions or need information on a different aspect of the Mars mission, feel free to ask!

Q: What challenges does the crew face with radiation?


2026-04-08 21:35:25,868 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:35:26,237 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:35:29,353 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The crew faces significant challenges related to radiation exposure on Mars. Unlike Earth, Mars lacks a global magnetic field, which makes its surface vulnerable to cosmic rays. Although the habitat is equipped with some shielding to mitigate this risk, long-term exposure to radiation remains a critical area of study. Dr. Hannah Weiss is actively monitoring the crew's health, conducting daily assessments and experiments to understand the effects of reduced gravity and increased radiation on the human body [Section: Living on Mars: Challenges and Adaptation].


## Step 7: Hybrid RAG with Reranking

The vanilla RAG above uses only **dense vector retrieval**. We can improve it by combining:

1. **Sparse retrieval (BM25)** — great for exact keyword / term matching
2. **Dense retrieval (vector)** — great for semantic / paraphrase matching
3. **Ensemble** — merges both result lists with configurable weights
4. **Reranking** — uses a cross-encoder to re-score and reorder the combined results

This is what production RAG systems typically look like.

In [34]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank

# --- BM25 (sparse) retriever over the same documents ---
bm25_retriever = BM25Retriever.from_documents(documents, k=10)
logger.info("BM25 sparse retriever created over %d documents", len(documents))

# --- Reuse dense (vector) retriever from Step 3, fetch more candidates for reranking ---
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# --- Ensemble: combine BM25 + dense with 40/60 weighting ---
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6],
)
logger.info("Ensemble retriever created (BM25=0.4, dense=0.6)")

# --- Reranker: FlashRank cross-encoder re-scores the ensemble output ---
reranker = FlashrankRerank(top_n=5)
hybrid_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=ensemble_retriever,
)
logger.info("Hybrid retriever with FlashRank reranking ready (top_n=5)")

# Quick comparison: same query through vanilla vs hybrid
test_query = "Who was the commander of the mission?"
print("=== Dense-only retriever (top 5) ===")
for i, doc in enumerate(retriever.invoke(test_query)[:5]):
    print(f"  [{i+1}] {doc.page_content}...\n")

print("=== Hybrid retriever with reranking (top 5) ===")
for i, doc in enumerate(hybrid_retriever.invoke(test_query)[:5]):
    print(f"  [{i+1}] {doc.page_content}...\n")

2026-04-08 21:52:06,098 [INFO] rag_lab - BM25 sparse retriever created over 23 documents
2026-04-08 21:52:06,100 [INFO] rag_lab - Ensemble retriever created (BM25=0.4, dense=0.6)
2026-04-08 21:52:06,505 [INFO] rag_lab - Hybrid retriever with FlashRank reranking ready (top_n=5)


=== Dense-only retriever (top 5) ===


2026-04-08 21:52:06,905 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


  [1] ---
### The Crew: Six Pioneers of a New Frontier
The *Aurora-1* crew consists of six individuals, each representing a different discipline and, in many ways, a different facet of humanity itself.
* **Commander Elena Navarro (Spain)** — A veteran astronaut and aerospace engineer, Navarro has led multiple lunar missions. Known for her calm precision, she was the natural choice to command humanity’s first Martian landing....

  [2] ---
### The Mission: A Journey Decades in the Making
The mission, officially named *Aurora-1*, is the culmination of over thirty years of international collaboration, technological innovation, and relentless ambition. Unlike previous robotic missions, which paved the way with data and imagery, *Aurora-1* represents humanity’s first physical presence on another planet....

  [3] * **Dr. Malik Okoye (Nigeria)** — Astrobiologist and mission science lead. His research focuses on extremophiles—organisms that survive in harsh environments—making him uniquely su

2026-04-08 21:52:07,263 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


  [1] The chosen landing site, Jezero Crater, was selected for its scientific significance. Once believed to be an ancient lakebed, it offers the highest probability of uncovering evidence of past microbial life. For scientists back on Earth, this mission is not only about exploration—it is about answering one of humanity’s oldest questions: Are we alone?
---
### The Crew: Six Pioneers of a New Frontier...

  [2] ---
### The Crew: Six Pioneers of a New Frontier
The *Aurora-1* crew consists of six individuals, each representing a different discipline and, in many ways, a different facet of humanity itself.
* **Commander Elena Navarro (Spain)** — A veteran astronaut and aerospace engineer, Navarro has led multiple lunar missions. Known for her calm precision, she was the natural choice to command humanity’s first Martian landing....

  [3] ---
### The Mission: A Journey Decades in the Making
The mission, officially named *Aurora-1*, is the culmination of over thirty years of internationa

In [35]:
hybrid_retriever_tool = create_retriever_tool(
    hybrid_retriever,
    name="mars_report_hybrid_search",
    description=(
        "Search the Mars landing report using hybrid retrieval (keyword + semantic + reranking). "
        "Use this tool to find facts before answering any question about the Aurora-1 mission."
    ),
)

hybrid_agent = create_agent(
    model=llm,
    tools=[hybrid_retriever_tool],
    system_prompt=SYSTEM_PROMPT,
)

logger.info("Hybrid RAG agent created")

# Run the same test questions from Step 4 so we can visually compare
for question in test_questions:
    print(f"\n{'=' * 70}")
    print(f"Q: {question}")
    print(f"{'=' * 70}")
    response = hybrid_agent.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content
    print(f"A: {answer}")

2026-04-08 21:53:28,353 [INFO] rag_lab - Hybrid RAG agent created



Q: Who was the first person to step on Mars?


2026-04-08 21:53:30,212 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:53:30,857 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:53:32,502 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The retrieved information does not specify who the first person to step on Mars was. Therefore, I cannot provide that detail.

Q: What was the name of the mission and when did it launch?


2026-04-08 21:53:33,639 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:53:33,955 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:53:35,423 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

A: The *Aurora-1* mission crew consists of six members. One of the crew members is Commander Elena Navarro from Spain. However, the retrieved information does not specify the nationalities of the other crew members.

Q: What scientific discoveries were made on Mars?


2026-04-08 21:53:41,300 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:53:41,636 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:53:45,102 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The Mars landing mission, particularly at Jezero Crater, has focused on significant scientific discoveries related to the search for past microbial life. The crater is believed to have once been an ancient lakebed, making it a prime location for uncovering evidence of life. 

Key scientists involved in the mission include:

- **Dr. Malik Okoye**, an astrobiologist, who is particularly interested in extremophiles—organisms that thrive in harsh environments—making him well-suited to search for signs of life on Mars.
- **Dr. Aiko Tanaka**, a geologist, whose expertise in sedimentary formations will aid in exploring the geological history of the crater.

The mission aims to answer fundamental questions about the existence of life beyond Earth, and the crew has begun collecting samples for analysis, which will be crucial for understanding Mars' past.

Q: What challenges did the crew face living on Mars?


2026-04-08 21:53:46,234 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:53:46,561 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:53:49,914 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The crew faced several challenges while living on Mars, including:

1. **Extreme Temperature Variations**: Temperatures on Mars can swing dramatically, with relatively mild conditions during the day and extreme cold at night.

2. **Dust Storms**: Although the crew had not yet encountered dust storms, these remain a persistent threat that could impact their operations.

3. **Terrain Difficulties**: The Martian terrain consists of fine dust and jagged rocks, which can complicate movement and exploration.

4. **Bulky Space Suits**: The crew had to wear bulky suits designed to protect against extreme temperatures, radiation, and the near-vacuum atmosphere, which constrained their movements and required deliberate actions.

5. **Isolation and Silence**: The environment is characterized by overwhelming silence, with no wind or familiar sounds, contributing to a sense of isolation.

These challenges necessitate careful planning and adaptation as the crew conducts their mission on Mars.


## Step 8: Comprehensive Evaluation — Vanilla vs Hybrid

Re-run the evaluation with **additional retrieval metrics** and compare both pipelines side-by-side.

| Metric | What it measures |
|---|---|
| **Hit Rate** | Did *any* relevant chunk appear in top-k? |
| **MRR** | How high was the *first* relevant chunk ranked? |
| **Precision@k** | What fraction of the top-k results are relevant? |
| **Recall@k** | What fraction of *all* relevant chunks appear in top-k? |
| **nDCG@k** | Are relevant chunks ranked higher than irrelevant ones? (position-aware) |
| **Faithfulness** | Does the answer only use info from retrieved context? (LLM judge, 1-5) |
| **Correctness** | Does the answer match the ground truth? (LLM judge, 1-5) |

In [36]:
import math


def compute_ndcg(relevance_scores: list[int], k: int) -> float:
    """Compute nDCG@k from a list of binary relevance labels."""
    dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance_scores[:k]))
    ideal = sorted(relevance_scores, reverse=True)[:k]
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_rag_pipeline(
    eval_data: list[dict],
    ret,
    ag,
    all_docs: list[Document],
    judge: ChatOpenAI,
    k: int = 5,
    label: str = "",
) -> list[dict]:
    """Run full retrieval + generation evaluation and return per-question metrics."""
    results = []

    for idx, item in enumerate(eval_data):
        question = item["question"]
        ground_truth = item["ground_truth"]
        keywords = item["keywords"]

        # --- Retrieval ---
        retrieved_docs = ret.invoke(question)[:k]
        retrieved_texts = [doc.page_content for doc in retrieved_docs]
        combined_context = "\n---\n".join(retrieved_texts)

        relevance = [
            1 if any(kw.lower() in txt.lower() for kw in keywords) else 0
            for txt in retrieved_texts
        ]

        total_relevant_in_corpus = sum(
            1 for doc in all_docs
            if any(kw.lower() in doc.page_content.lower() for kw in keywords)
        )

        hit = 1 if any(r == 1 for r in relevance) else 0

        rr = 0.0
        for rank, rel in enumerate(relevance, start=1):
            if rel == 1:
                rr = 1.0 / rank
                break

        precision_k = sum(relevance) / k if k > 0 else 0.0
        recall_k = sum(relevance) / total_relevant_in_corpus if total_relevant_in_corpus > 0 else 0.0
        ndcg_k = compute_ndcg(relevance, k)

        # --- Generation ---
        response = ag.invoke({"messages": [{"role": "user", "content": question}]})
        answer = response["messages"][-1].content

        # --- LLM-as-judge ---
        judge_prompt = JUDGE_TEMPLATE.format(
            question=question,
            ground_truth=ground_truth,
            context=combined_context[:3000],
            answer=answer,
        )
        judge_resp = judge.invoke(judge_prompt)
        judge_text = judge_resp.content

        faith, correct = 0, 0
        for line in judge_text.strip().split("\n"):
            if line.lower().startswith("faithfulness"):
                try:
                    faith = int(line.split(":")[1].strip())
                except (ValueError, IndexError):
                    pass
            elif line.lower().startswith("correctness"):
                try:
                    correct = int(line.split(":")[1].strip())
                except (ValueError, IndexError):
                    pass

        results.append({
            "question": question,
            "answer": answer,
            "hit": hit,
            "rr": rr,
            "precision_k": precision_k,
            "recall_k": recall_k,
            "ndcg_k": ndcg_k,
            "faithfulness": faith,
            "correctness": correct,
        })

        logger.info(
            "[%s %d/%d] %s | hit=%d p@k=%.2f r@k=%.2f ndcg=%.2f faith=%d corr=%d",
            label, idx + 1, len(eval_data), question[:30],
            hit, precision_k, recall_k, ndcg_k, faith, correct,
        )

    return results


logger.info("Evaluation function ready")

2026-04-08 21:54:08,939 [INFO] rag_lab - Evaluation function ready


In [37]:
logger.info("Running evaluation on VANILLA (dense-only) pipeline...")
vanilla_eval = evaluate_rag_pipeline(
    eval_dataset, retriever, agent, documents, judge_llm, k=5, label="vanilla",
)

logger.info("Running evaluation on HYBRID (BM25 + dense + rerank) pipeline...")
hybrid_eval = evaluate_rag_pipeline(
    eval_dataset, hybrid_retriever, hybrid_agent, documents, judge_llm, k=5, label="hybrid",
)

logger.info("Both evaluations complete")

2026-04-08 21:54:37,284 [INFO] rag_lab - Running evaluation on VANILLA (dense-only) pipeline...
2026-04-08 21:54:37,913 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:54:39,409 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:54:39,729 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:54:41,223 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:54:42,453 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 21:54:42,454 [INFO] rag_lab - [vanilla 1/10] Who was the commander of the A | hit=1 p@k=0.20 r@k=0.50 ndcg=1.00 faith=5 corr=5
2026-04-08 21:54:42,664 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-08 21:54:43,500 [INFO] httpx - HTTP Request: POST https://api.

In [38]:
def summarize(results: list[dict]) -> dict:
    n = len(results)
    return {
        "Hit Rate": sum(r["hit"] for r in results) / n,
        "MRR": sum(r["rr"] for r in results) / n,
        "Precision@k": sum(r["precision_k"] for r in results) / n,
        "Recall@k": sum(r["recall_k"] for r in results) / n,
        "nDCG@k": sum(r["ndcg_k"] for r in results) / n,
        "Faithfulness": sum(r["faithfulness"] for r in results) / n,
        "Correctness": sum(r["correctness"] for r in results) / n,
    }


vanilla_summary = summarize(vanilla_eval)
hybrid_summary = summarize(hybrid_eval)

# --- Per-question comparison ---
print(f"\n{'=' * 100}")
print(f"{'PER-QUESTION COMPARISON':^100}")
print(f"{'=' * 100}")
header = f"{'Question':<40} | {'Hit':^7} | {'P@k':^11} | {'R@k':^11} | {'nDCG':^11} | {'Corr':^9}"
print(f"\n{header}")
print(f"{'-' * 40}-+-{'-' * 7}-+-{'-' * 11}-+-{'-' * 11}-+-{'-' * 11}-+-{'-' * 9}")

for v, h in zip(vanilla_eval, hybrid_eval):
    q = (v["question"][:37] + "...") if len(v["question"]) > 40 else v["question"]
    print(
        f"{q:<40} | "
        f"{'V' if v['hit'] else '-':>1} / {'H' if h['hit'] else '-':>1}   | "
        f"{v['precision_k']:.2f}/{h['precision_k']:.2f}  | "
        f"{v['recall_k']:.2f}/{h['recall_k']:.2f}  | "
        f"{v['ndcg_k']:.2f}/{h['ndcg_k']:.2f}  | "
        f"{v['correctness']}/{h['correctness']}      "
    )

# --- Aggregate comparison ---
print(f"\n\n{'=' * 60}")
print(f"{'AGGREGATE COMPARISON':^60}")
print(f"{'=' * 60}")
print(f"\n{'Metric':<20} {'Vanilla':>12} {'Hybrid':>12} {'Delta':>10}")
print(f"{'-' * 20} {'-' * 12} {'-' * 12} {'-' * 10}")

for metric in vanilla_summary:
    v_val = vanilla_summary[metric]
    h_val = hybrid_summary[metric]
    delta = h_val - v_val
    sign = "+" if delta > 0 else ""
    if metric in ("Faithfulness", "Correctness"):
        print(f"{metric:<20} {v_val:>10.2f}/5 {h_val:>10.2f}/5 {sign}{delta:>8.2f}")
    else:
        print(f"{metric:<20} {v_val:>11.1%} {h_val:>11.1%} {sign}{delta:>8.1%}")

print(f"{'=' * 60}")
print(f"\n  Chunking strategy used: {CHUNKING_STRATEGY}")
print(f"  Vanilla = dense vector retrieval only")
print(f"  Hybrid  = BM25 + dense ensemble + FlashRank reranking")


                                      PER-QUESTION COMPARISON                                       

Question                                 |   Hit   |     P@k     |     R@k     |    nDCG     |   Corr   
-----------------------------------------+---------+-------------+-------------+-------------+----------
Who was the commander of the Aurora-1... | V / H   | 0.20/0.20  | 0.50/0.50  | 1.00/0.43  | 5/5      
When did humans first land on Mars?      | V / H   | 0.20/0.20  | 1.00/1.00  | 1.00/0.39  | 5/5      
What was the name of the lander?         | V / H   | 0.40/0.40  | 1.00/1.00  | 0.92/0.92  | 5/5      
What is the gravity on Mars compared ... | V / H   | 0.20/0.20  | 1.00/1.00  | 1.00/1.00  | 5/5      
What was Dr. Malik Okoye's role on th... | V / H   | 0.20/0.20  | 1.00/1.00  | 1.00/0.50  | 5/5      
What were the early scientific findin... | V / -   | 0.20/0.00  | 1.00/0.00  | 0.39/0.00  | 5/3      
Where did the Aurora-1 mission land o... | V / -   | 0.20/0.00  | 0.33/0.00

## TODO

- [ ] **Log evaluation metrics to LangSmith / Langfuse** — Integrate with an observability platform so you can track retrieval and generation metrics across runs, compare experiments, and share results with your team.
- [ ] **Try with other methods to see the difference** — Swap out components (e.g. different chunking strategies, embedding models, distance metrics, retriever weights, reranker models) and re-run the evaluation to understand how each choice affects quality.
- [ ] **Add more test cases to see how the RAG performs** — Expand `eval_dataset` with harder questions (multi-hop, negation, out-of-scope) to stress-test retrieval and generation quality.
- [ ] **Add more document to RAG** - Add spaceship doc to the knowledge and try to ask some more questions related to that doc to see how the agent will answer.